# **Bag of Words (BOW) Text Classification**

This notebook demonstrates how to use TFIDF Vectorizer from scikit-learn

_______________

### **Set-Up**

This section focuses on loading the data, python libraries (which will be used throughout this notebook)

In [3]:
# Loading Libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("--"*50)
print("Libraries loaded successfully.")
print("--"*50)

----------------------------------------------------------------------------------------------------
Libraries loaded successfully.
----------------------------------------------------------------------------------------------------


In [4]:
# Loading the dataset
data = pd.read_csv("data/dataset_with_assignments.csv")

print("--"*50)
print("First few Rows of the Dataset:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
First few Rows of the Dataset:
----------------------------------------------------------------------------------------------------


,page_id,url,domain,tld,date,word_count,text_length,sentence_count,paragraph_count,avg_word_length,path_depth,text,full_text,assigned_to,manual_label
0,1,http://0769sme.org/index-16.html,0769sme.org,org,2025-12-04T20:51:02Z,2377,14900,150,1,6.27,1,Best Resume Templates 2024 | Ready to Download...,Best Resume Templates 2024 | Ready to Download...,Gaurav Advani,NaN
1,4,http://aastocks.com/en/cnhk/quote/quick-quote....,aastocks.com,com,2025-12-04T21:23:33Z,1738,12165,73,1,7.00,4,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,Gaurav Advani,NaN
2,6,http://ada.untergrund.net/?p=boardthread&id=18...,ada.untergrund.net,net,2025-12-04T20:53:54Z,3759,20609,270,1,5.48,0,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,Gaurav Advani,NaN
3,7,http://adrienedurand.wikidot.com/blog:127,adrienedurand.wikidot.com,com,2025-12-04T21:11:53Z,1328,7829,73,1,5.90,1,"Time, Mortality And Memory - blog from trends\...","Time, Mortality And Memory - blog from trends\...",Gaurav Advani,NaN
4,8,http://afrafrontpagenews.blogspot.com/2012/03/...,afrafrontpagenews.blogspot.com,com,2025-12-04T19:58:47Z,5411,31154,654,1,5.76,3,AFR News: JurisDictionary- How Your Constituti...,AFR News: JurisDictionary- How Your Constituti...,Gaurav Advani,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6309,8259,https://zettelfamilyfarms.ca/april-2022-newsle...,zettelfamilyfarms.ca,ca,2025-12-04T20:18:59Z,1471,8401,94,1,5.71,1,April 2022 Newsletter – Zettel Family Farms\nS...,April 2022 Newsletter – Zettel Family Farms\nS...,NaN,NaN
6310,8261,https://ziloqa.net/ponds-bright-beauty-spot-le...,ziloqa.net,net,2025-12-04T20:37:36Z,343,2151,18,1,6.27,1,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,NaN,NaN
6311,8262,https://ziloqa.net/understanding-bitcoin-and-i...,ziloqa.net,net,2025-12-04T21:14:36Z,563,3782,74,1,6.72,1,Understanding Bitcoin And Its Market Trends Ho...,Understanding Bitcoin And Its Market Trends Ho...,NaN,NaN
6312,8264,https://zmanswoodart.com/handcrafted-wooden-fi...,zmanswoodart.com,com,2025-12-04T20:39:34Z,252,1451,10,1,5.76,1,Woman and Dog Wood Figurine | Z Man's Wood Art...,Woman and Dog Wood Figurine | Z Man's Wood Art...,NaN,NaN


In [5]:
# Dataset Shape:
print("--"*50)
print("Dataset Shape (Rows, Columns)")
print("--"*50)
data.shape

----------------------------------------------------------------------------------------------------
Dataset Shape (Rows, Columns)
----------------------------------------------------------------------------------------------------


(6314, 15)

In [6]:
# Dataset Information
print("--"*50)
print("Dataset Information")
print("--"*50)
data.info()

----------------------------------------------------------------------------------------------------
Dataset Information
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6314 entries, 0 to 6313
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   page_id          6314 non-null   int64  
 1   url              6314 non-null   object 
 2   domain           6314 non-null   object 
 3   tld              6314 non-null   object 
 4   date             6314 non-null   object 
 5   word_count       6314 non-null   int64  
 6   text_length      6314 non-null   int64  
 7   sentence_count   6314 non-null   int64  
 8   paragraph_count  6314 non-null   int64  
 9   avg_word_length  6314 non-null   float64
 10  path_depth       6314 non-null   int64  
 11  text             6314 non-null   object 
 12  full_text        6314 

In [7]:
# Identifying NULLs and NaNs in the data set
print("--"*50)
print("Null Values in the dataset")
print("--"*50)
print(data.isnull().sum())

----------------------------------------------------------------------------------------------------
Null Values in the dataset
----------------------------------------------------------------------------------------------------
page_id               0
url                   0
domain                0
tld                   0
date                  0
word_count            0
text_length           0
sentence_count        0
paragraph_count       0
avg_word_length       0
path_depth            0
text                  0
full_text             0
assigned_to        5514
manual_label       6314
dtype: int64


In [8]:
print("--"*50)
print("Data Description for Numeric features:")
print("--"*50)
data.describe(include=[np.number])

----------------------------------------------------------------------------------------------------
Data Description for Numeric features:
----------------------------------------------------------------------------------------------------


,page_id,word_count,text_length,sentence_count,paragraph_count,avg_word_length,path_depth,manual_label
count,6314.000000,6314.000000,6.314000e+03,6314.000000,6314.000000,6314.000000,6314.000000,0.0
mean,4196.268609,1029.227114,7.092383e+03,59.067152,1.128286,7.040988,2.355084,NaN
std,2372.410287,2247.213005,2.990320e+04,427.019873,2.290003,12.457661,1.475493,NaN
min,1.000000,51.000000,2.590000e+02,1.000000,1.000000,3.490000,0.000000,NaN
25%,2140.250000,277.000000,1.809000e+03,11.000000,1.000000,6.100000,1.000000,NaN
50%,4274.500000,584.000000,3.785000e+03,24.000000,1.000000,6.460000,2.000000,NaN
75%,6241.750000,1149.000000,7.556250e+03,51.000000,1.000000,6.880000,3.000000,NaN
max,8268.000000,86776.000000,2.099836e+06,31928.000000,117.000000,488.220000,29.000000,NaN


Based on the above information, it is clear from a bird’s-eye view that there are no null or NaN (except for manual_label) values or even duplication of rows within the dataset.

____________

### **1. Text Preprocessing**

In this section, the primary focus is on cleaning the text data.Identifying and removing these characters early is important, as they can cause issues later in the pipeline and can negatively impact stability and compatibility. The primary goal here is to retain ASCII characters, such as English letters, numbers, and common punctuation (e.g., '(', ')', '[', ']', '+', '-', ' '), while ensuring that the existing structure of the text remains unchanged.

In [9]:
# list of currency symbols (few of them might not be present, but still we will keep it)
currency_symbols = [
    # Paired (Multi-character) symbols (Longest first)
    'USD','AU$', 'C$', 'NZ$', 'HK$', 'S$', 'US$', 'NT$', 'MX$', 'R$', 'zł', 'Kč','₨',
    'kr', 'Ft', 'ден', 'р.', 'лв', '₡', '₢', '₣', '₥', 

    # Single character symbols (Specific before generic)
    '€', '£', '¥', '元', '₹', '฿', '₩', '₽', '₪', '₺', '₦', '₵', '₫', 
    '₱', '₭', '₮', '₼', '₸', '₴', '៛', '₲', '₡', '₾', '֏', '﷼', 'Ξ', 'Ł', '$'
]

curr_symbols = ''.join(re.escape(s) for s in currency_symbols)
# function to clean the full text column
def full_text_clean (text):
    #Regex Pattern to indentify all the ASCII characters while retaining currency symbols
    re_pattern = rf"[\x00-\x7F{curr_symbols}]+"

    clean = re.findall(re_pattern, text)

    return clean

In [10]:
data['full_text_clean'] = data['full_text'].apply(full_text_clean)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,url,domain,tld,date,word_count,text_length,sentence_count,paragraph_count,avg_word_length,path_depth,text,full_text,assigned_to,manual_label,full_text_clean
0,1,http://0769sme.org/index-16.html,0769sme.org,org,2025-12-04T20:51:02Z,2377,14900,150,1,6.27,1,Best Resume Templates 2024 | Ready to Download...,Best Resume Templates 2024 | Ready to Download...,Gaurav Advani,NaN,[Best Resume Templates 2024 | Ready to Downloa...
1,4,http://aastocks.com/en/cnhk/quote/quick-quote....,aastocks.com,com,2025-12-04T21:23:33Z,1738,12165,73,1,7.00,4,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,Gaurav Advani,NaN,"[SH/SZ-HK Stock Connect Quick Quote\n, \n, \nM..."
2,6,http://ada.untergrund.net/?p=boardthread&id=18...,ada.untergrund.net,net,2025-12-04T20:53:54Z,3759,20609,270,1,5.48,0,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,Gaurav Advani,NaN,[A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga ...
3,7,http://adrienedurand.wikidot.com/blog:127,adrienedurand.wikidot.com,com,2025-12-04T21:11:53Z,1328,7829,73,1,5.90,1,"Time, Mortality And Memory - blog from trends\...","Time, Mortality And Memory - blog from trends\...",Gaurav Advani,NaN,"[Time, Mortality And Memory - blog from trends..."
4,8,http://afrafrontpagenews.blogspot.com/2012/03/...,afrafrontpagenews.blogspot.com,com,2025-12-04T19:58:47Z,5411,31154,654,1,5.76,3,AFR News: JurisDictionary- How Your Constituti...,AFR News: JurisDictionary- How Your Constituti...,Gaurav Advani,NaN,[AFR News: JurisDictionary- How Your Constitut...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6309,8259,https://zettelfamilyfarms.ca/april-2022-newsle...,zettelfamilyfarms.ca,ca,2025-12-04T20:18:59Z,1471,8401,94,1,5.71,1,April 2022 Newsletter – Zettel Family Farms\nS...,April 2022 Newsletter – Zettel Family Farms\nS...,NaN,NaN,"[April 2022 Newsletter , Zettel Family Farms\..."
6310,8261,https://ziloqa.net/ponds-bright-beauty-spot-le...,ziloqa.net,net,2025-12-04T20:37:36Z,343,2151,18,1,6.27,1,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,NaN,NaN,[PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 5...
6311,8262,https://ziloqa.net/understanding-bitcoin-and-i...,ziloqa.net,net,2025-12-04T21:14:36Z,563,3782,74,1,6.72,1,Understanding Bitcoin And Its Market Trends Ho...,Understanding Bitcoin And Its Market Trends Ho...,NaN,NaN,[Understanding Bitcoin And Its Market Trends H...
6312,8264,https://zmanswoodart.com/handcrafted-wooden-fi...,zmanswoodart.com,com,2025-12-04T20:39:34Z,252,1451,10,1,5.76,1,Woman and Dog Wood Figurine | Z Man's Wood Art...,Woman and Dog Wood Figurine | Z Man's Wood Art...,NaN,NaN,[Woman and Dog Wood Figurine | Z Man's Wood Ar...


#### **Replacing `\n` with Space**

Replacing `\n` with a space removes the ambiguity caused when text appears on a new line but is recorded as `\n`, resulting in a single continuous text line.

In [11]:
# Cleaning "\n" from the cleaned full_text column
def clean_text(text):
    clean = r"[\r\n]+"

    sentence = []

    # Find all matches
    for i in range (len(text)):
        sentence.append(re.sub(clean, " ", text[i]))

    sentence = [stn for stn in sentence if stn]

    return sentence

data['full_text_clean'] = data['full_text_clean'].apply(clean_text)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,url,domain,tld,date,word_count,text_length,sentence_count,paragraph_count,avg_word_length,path_depth,text,full_text,assigned_to,manual_label,full_text_clean
0,1,http://0769sme.org/index-16.html,0769sme.org,org,2025-12-04T20:51:02Z,2377,14900,150,1,6.27,1,Best Resume Templates 2024 | Ready to Download...,Best Resume Templates 2024 | Ready to Download...,Gaurav Advani,NaN,[Best Resume Templates 2024 | Ready to Downloa...
1,4,http://aastocks.com/en/cnhk/quote/quick-quote....,aastocks.com,com,2025-12-04T21:23:33Z,1738,12165,73,1,7.00,4,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,SH/SZ-HK Stock Connect Quick Quote\n繁\n简\nMark...,Gaurav Advani,NaN,"[SH/SZ-HK Stock Connect Quick Quote , , Mark..."
2,6,http://ada.untergrund.net/?p=boardthread&id=18...,ada.untergrund.net,net,2025-12-04T20:53:54Z,3759,20609,270,1,5.48,0,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,A.D.A. Amiga Demoscene Archive\nA.D.A. Amiga D...,Gaurav Advani,NaN,[A.D.A. Amiga Demoscene Archive A.D.A. Amiga D...
3,7,http://adrienedurand.wikidot.com/blog:127,adrienedurand.wikidot.com,com,2025-12-04T21:11:53Z,1328,7829,73,1,5.90,1,"Time, Mortality And Memory - blog from trends\...","Time, Mortality And Memory - blog from trends\...",Gaurav Advani,NaN,"[Time, Mortality And Memory - blog from trends..."
4,8,http://afrafrontpagenews.blogspot.com/2012/03/...,afrafrontpagenews.blogspot.com,com,2025-12-04T19:58:47Z,5411,31154,654,1,5.76,3,AFR News: JurisDictionary- How Your Constituti...,AFR News: JurisDictionary- How Your Constituti...,Gaurav Advani,NaN,[AFR News: JurisDictionary- How Your Constitut...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6309,8259,https://zettelfamilyfarms.ca/april-2022-newsle...,zettelfamilyfarms.ca,ca,2025-12-04T20:18:59Z,1471,8401,94,1,5.71,1,April 2022 Newsletter – Zettel Family Farms\nS...,April 2022 Newsletter – Zettel Family Farms\nS...,NaN,NaN,"[April 2022 Newsletter , Zettel Family Farms ..."
6310,8261,https://ziloqa.net/ponds-bright-beauty-spot-le...,ziloqa.net,net,2025-12-04T20:37:36Z,343,2151,18,1,6.27,1,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 50...,NaN,NaN,[PONDS BRIGHT BEAUTY SPOT-LESS GLOW FACEWASH 5...
6311,8262,https://ziloqa.net/understanding-bitcoin-and-i...,ziloqa.net,net,2025-12-04T21:14:36Z,563,3782,74,1,6.72,1,Understanding Bitcoin And Its Market Trends Ho...,Understanding Bitcoin And Its Market Trends Ho...,NaN,NaN,[Understanding Bitcoin And Its Market Trends H...
6312,8264,https://zmanswoodart.com/handcrafted-wooden-fi...,zmanswoodart.com,com,2025-12-04T20:39:34Z,252,1451,10,1,5.76,1,Woman and Dog Wood Figurine | Z Man's Wood Art...,Woman and Dog Wood Figurine | Z Man's Wood Art...,NaN,NaN,[Woman and Dog Wood Figurine | Z Man's Wood Ar...


#### **Removing Few Common Words**

Removal of common words such as "Facebook", "YouTube", "Home", "About", etc., words that do not add meaningful value to the text, but are neighter stop words

In [15]:
# Random index number to check how a full_text field and full_text_clean field looks like
idx = np.random.randint(0, 6314)
char_len = 0
for i in range (len(data.loc[idx,'full_text_clean'])):
    char_len += len(data.loc[idx,'full_text_clean'][i])

print("--"*50)
print(f"Full Text Clean Field ({idx})-- Length({len(data.loc[idx,'full_text_clean'][0:10000])}) -- Total Characters Length: {char_len}")
print("--"*50)
print(data.loc[idx, 'full_text_clean'][0:5])

----------------------------------------------------------------------------------------------------
Full Text Clean Field (5358)-- Length(50) -- Total Characters Length: 16410
----------------------------------------------------------------------------------------------------
['Lessons From 30 Years of Government Reform Efforts - Nextgov/FCW Continue to the site', 'Skip to Content Notice at Collection Your Privacy Choices Exercise Your Privacy Rights SAP offers major discount to government customers through OneGov agreement New bill proposes government-wide processes to attribute, sanction hackers AWS announces new AI Factories to reduce infrastructure barriers for public, private sector Democrats bring back AI civil rights bill Nominations are open for the 2026 Fed100 awards sponsor content Government AI hits a data roadblock but synthetic data could be the fix SAP offers major discount to government customers through OneGov agreement New bill proposes government-wide processes to at

#### **Removal of Stop Word**

In this step, we remove stop words. Word tokens that contribute minimal semantic or discriminatory value to the classification objective when considered independently.

In [16]:
idx = np.random.randint(0, 6314)

print("--"*50)
print(f"Original Full Text Field ({idx}) -- Length({len(data.loc[idx,'full_text_clean'][0:])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'][0:5])

----------------------------------------------------------------------------------------------------
Original Full Text Field (4017) -- Length(26)
----------------------------------------------------------------------------------------------------
['KRMF706EBS by KitchenAid - 25.8 Cu. Ft. 36" Multi-Door Freestanding Refrigerator with Platinum Interior Design and PrintShield', 'Finish | Bousquet Appliance Skip disability assistance statement. Welcome to our website! As we have the ability to list over one million items on our website (our selection changes all of the time), it is not feasible for a company our size to record and playback the descriptions on every item on our website. However, if you have a disability we are here to help you. Please call our disability services phone line at 860-774-5821 during regular business hours and one of our kind and friendly personal shoppers will help you navigate through our website, help conduct advanced searches, help you choose the item you 

In [22]:
# Rmoving stoop words
def remove_stop_words(text):
    vocab = []

    for i in range (len(text)):
        # Initialize CountVectorizer with the English stop words list
        vectorizer = CountVectorizer(stop_words='english')
        # Fit and transform the text data
        X = vectorizer.fit_transform(text[i])
        vocab.append(vectorizer.vocabulary_)

    return vocab
    

In [23]:
# Applying the function to remove stop words from full_text_clean column
data['full_text_clean_nostop'] = data['full_text_clean'].apply(remove_stop_words)

print("--"*50)
print("New datastructure after removing stop words from full_text_clean column:")
print("--"*50)
data['full_text_clean_nostop']

ValueError: Iterable over raw text documents expected, string object received.

In [ ]:
data[['page_id','full_text_clean','full_text_clean_nostop']].to_csv("data/dataset_with_part1.csv", index=False)